In [2]:
import sympy;
from IPython.display import display, Math
from sympy.printing.octave import octave_code, print_octave_code
from sympy.utilities.codegen import codegen
sympy.init_printing()
u, v, m, n, e, rho, gamma, p, c = sympy.symbols('u v m n e rho gamma P c')
U = sympy.Matrix([rho, m, n, e])



In [3]:
P = (gamma - 1)*(e - (m*m+n*n)/(2*rho))

F = sympy.Matrix([m, m*m/rho + P, m*n/rho, (e+P)*m/rho])
A = F.jacobian(U).subs(P, p).subs((gamma-1)*(m*m+n*n)/(2*rho), (gamma-1)*e - p)
A.simplify()
display(A)

⎡            0                         1                   0         0  ⎤
⎢                                                                       ⎥
⎢   2                                                                   ⎥
⎢- m  + ρ⋅(-P + e⋅(γ - 1))         m⋅(3 - γ)           n⋅(1 - γ)        ⎥
⎢─────────────────────────         ─────────           ─────────   γ - 1⎥
⎢            2                         ρ                   ρ            ⎥
⎢           ρ                                                           ⎥
⎢                                                                       ⎥
⎢          -m⋅n                        n                   m            ⎥
⎢          ─────                       ─                   ─         0  ⎥
⎢            2                         ρ                   ρ            ⎥
⎢           ρ                                                           ⎥
⎢                                                                       ⎥
⎢                              2      

In [4]:
G = sympy.Matrix([n, m*n/rho, n*n/rho + P, (e+P)*n/rho])
B = G.jacobian(U).subs(P, p).subs((gamma-1)*(m*m+n*n)/(2*rho), (gamma-1)*e - p)
B.simplify()
B

⎡            0                   0                  1                0  ⎤
⎢                                                                       ⎥
⎢          -m⋅n                  n                  m                   ⎥
⎢          ─────                 ─                  ─                0  ⎥
⎢            2                   ρ                  ρ                   ⎥
⎢           ρ                                                           ⎥
⎢                                                                       ⎥
⎢   2                                                                   ⎥
⎢- n  + ρ⋅(-P + e⋅(γ - 1))   m⋅(1 - γ)          n⋅(3 - γ)               ⎥
⎢─────────────────────────   ─────────          ─────────          γ - 1⎥
⎢            2                   ρ                  ρ                   ⎥
⎢           ρ                                                           ⎥
⎢                                                                       ⎥
⎢                                     

The eigenvectors of $A$ are $u$ (multiplicity 2), $u+c$, and $u-c$.  So, the spectral radius is $|u| + c$, or equivalently $\frac{|m|} \rho + \sqrt{\frac{\gamma P} \rho}$.

Similarly, the spectral radius of $B$ is $\frac{|n|} \rho + \sqrt{\frac{\gamma P} \rho}$

In [5]:
dP = sympy.Matrix([P]).jacobian(U)
dP.simplify()
dP

⎡        ⎛ 2    2⎞                             ⎤
⎢(γ - 1)⋅⎝m  + n ⎠  m⋅(1 - γ)  n⋅(1 - γ)       ⎥
⎢─────────────────  ─────────  ─────────  γ - 1⎥
⎢         2             ρ          ρ           ⎥
⎣      2⋅ρ                                     ⎦

In [6]:
rho_xi, rho_eta, m_xi, m_eta, n_xi, n_eta, e_xi, e_eta = sympy.symbols('rho_xi rho_eta m_xi m_eta n_xi n_eta e_xi e_eta')
xi_x, xi_y, eta_x, eta_y, mu, T_x, T_y = sympy.symbols('xi_x xi_y eta_x eta_y mu T_x T_y')
k, R = sympy.symbols('k R')

u_xi = (rho*m_xi-m*rho_xi)/(rho*rho)
u_eta = (rho*m_eta - m*rho_eta)/(rho*rho)
v_xi = (rho*n_xi-n*rho_xi)/(rho*rho)
v_eta = (rho*n_eta-n*rho_eta)/(rho*rho)

e_x = xi_x*e_xi + eta_x*e_eta
e_y = xi_y*e_xi + eta_y*e_eta
u_x = xi_x*u_xi + eta_x*u_eta
u_y = xi_y*u_xi + eta_y*u_eta
v_x = xi_x*v_xi + eta_x*v_eta
v_y = xi_y*v_xi + eta_y*v_eta
rho_x = xi_x*rho_xi + eta_x*rho_eta
rho_y = xi_y*rho_xi + eta_y*rho_eta

T = ((gamma-1)/R)*(e/rho - (m*m + n*n)/(2*rho*rho))
T_x_expr = ((gamma-1)/R)*((rho*e_x-e*rho_x)/(rho*rho) - u_x*(m/rho) - v_x*(n/rho))
T_y_expr = ((gamma-1)/R)*((rho*e_y-e*rho_y)/(rho*rho) - u_y*(m/rho) - v_y*(n/rho))

tau_xx = 2*mu*(2*u_x - v_y)/3
tau_yy = 2*mu*(2*v_y - u_x)/3
tau_xy = mu*(u_y + v_x)

F_v = sympy.Matrix([0, tau_xx, tau_xy, m*tau_xx/rho + n*tau_xy/rho + k*T_x_expr])
G_v = sympy.Matrix([0, tau_xy, tau_yy, m*tau_xy/rho + n*tau_yy/rho + k*T_y_expr])

U_xi = sympy.Matrix([rho_xi, m_xi, n_xi, e_xi])
U_eta = sympy.Matrix([rho_eta, m_eta, n_eta, e_eta])

dT = sympy.Matrix([T]).jacobian(U)
dT.simplify()
dT

⎡        ⎛        2    2⎞                             ⎤
⎢(γ - 1)⋅⎝-e⋅ρ + m  + n ⎠  m⋅(1 - γ)  n⋅(1 - γ)  γ - 1⎥
⎢────────────────────────  ─────────  ─────────  ─────⎥
⎢             3                 2          2      R⋅ρ ⎥
⎣          R⋅ρ               R⋅ρ        R⋅ρ           ⎦

In [7]:
dFv_dU_xi = F_v.jacobian(U_xi).subs(T_x_expr, T_x)
dFv_dU_xi.simplify()
dFv_dU_xi

⎡                                        0                                     ↪
⎢                                                                              ↪
⎢                              2⋅μ⋅(-2⋅m⋅ξₓ + n⋅ξ_y)                           ↪
⎢                              ─────────────────────                           ↪
⎢                                         2                                    ↪
⎢                                      3⋅ρ                                     ↪
⎢                                                                              ↪
⎢                               -μ⋅(m⋅ξ_y + n⋅ξₓ)                              ↪
⎢                               ──────────────────                             ↪
⎢                                        2                                     ↪
⎢                                       ρ                                      ↪
⎢                                                                              ↪
⎢  R⋅μ⋅(2⋅m⋅(2⋅m⋅ξₓ - n⋅ξ_y)

In [8]:
dFv_dU_eta = F_v.jacobian(U_eta)
dFv_dU_eta.simplify()
dFv_dU_eta

⎡                                        0                                     ↪
⎢                                                                              ↪
⎢                              2⋅μ⋅(-2⋅ηₓ⋅m + η_y⋅n)                           ↪
⎢                              ─────────────────────                           ↪
⎢                                         2                                    ↪
⎢                                      3⋅ρ                                     ↪
⎢                                                                              ↪
⎢                               -μ⋅(ηₓ⋅n + η_y⋅m)                              ↪
⎢                               ──────────────────                             ↪
⎢                                        2                                     ↪
⎢                                       ρ                                      ↪
⎢                                                                              ↪
⎢  R⋅μ⋅(2⋅m⋅(2⋅ηₓ⋅m - η_y⋅n)

In [9]:
dGv_dU_xi = G_v.jacobian(U_xi)
dGv_dU_xi.simplify()
dGv_dU_xi

⎡                                        0                                     ↪
⎢                                                                              ↪
⎢                               -μ⋅(m⋅ξ_y + n⋅ξₓ)                              ↪
⎢                               ──────────────────                             ↪
⎢                                        2                                     ↪
⎢                                       ρ                                      ↪
⎢                                                                              ↪
⎢                              2⋅μ⋅(m⋅ξₓ - 2⋅n⋅ξ_y)                            ↪
⎢                              ────────────────────                            ↪
⎢                                         2                                    ↪
⎢                                      3⋅ρ                                     ↪
⎢                                                                              ↪
⎢R⋅μ⋅(-3⋅m⋅(m⋅ξ_y + n⋅ξₓ) + 

In [10]:
dGv_dU_eta = G_v.jacobian(U_eta)
dGv_dU_eta.simplify()
dGv_dU_eta

⎡                                        0                                     ↪
⎢                                                                              ↪
⎢                               -μ⋅(ηₓ⋅n + η_y⋅m)                              ↪
⎢                               ──────────────────                             ↪
⎢                                        2                                     ↪
⎢                                       ρ                                      ↪
⎢                                                                              ↪
⎢                              2⋅μ⋅(ηₓ⋅m - 2⋅η_y⋅n)                            ↪
⎢                              ────────────────────                            ↪
⎢                                         2                                    ↪
⎢                                      3⋅ρ                                     ↪
⎢                                                                              ↪
⎢R⋅μ⋅(-3⋅m⋅(ηₓ⋅n + η_y⋅m) + 

In [11]:
dFv_dU = F_v.jacobian(U)
dFv_dU.simplify()
dFv_dU

⎡                                                                              ↪
⎢                                                                              ↪
⎢                                                                              ↪
⎢                                                                              ↪
⎢                                                                              ↪
⎢                                                                              ↪
⎢                                                                              ↪
⎢                                                                              ↪
⎢                                                                              ↪
⎢                                                                              ↪
⎢                                                                              ↪
⎢                                                                              ↪
⎢                           

In [12]:
dGv_dU = G_v.jacobian(U)
dGv_dU.simplify()
dGv_dU

⎡                                                                              ↪
⎢                                                                              ↪
⎢                                                                              ↪
⎢                                                                              ↪
⎢                                                                              ↪
⎢                                                                              ↪
⎢                                                                              ↪
⎢                                                                              ↪
⎢                                                                              ↪
⎢                                                                              ↪
⎢                                                                              ↪
⎢                                                                              ↪
⎢                           

In [13]:
print(octave_code(dFv_dU))


[0 0 0 0; 2*mu.*(4*eta_x.*m.*rho_eta - 2*eta_x.*m_eta.*rho - 2*eta_y.*n.*rho_eta + eta_y.*n_eta.*rho + 4*m.*rho_xi.*xi_x - 2*m_xi.*rho.*xi_x - 2*n.*rho_xi.*xi_y + n_xi.*rho.*xi_y)./(3*rho.^3) -4*mu.*(eta_x.*rho_eta + rho_xi.*xi_x)./(3*rho.^2) 2*mu.*(eta_y.*rho_eta + rho_xi.*xi_y)./(3*rho.^2) 0; mu.*(2*eta_x.*n.*rho_eta - eta_x.*n_eta.*rho + 2*eta_y.*m.*rho_eta - eta_y.*m_eta.*rho + 2*m.*rho_xi.*xi_y - m_xi.*rho.*xi_y + 2*n.*rho_xi.*xi_x - n_xi.*rho.*xi_x)./rho.^3 -mu.*(eta_y.*rho_eta + rho_xi.*xi_y)./rho.^2 -mu.*(eta_x.*rho_eta + rho_xi.*xi_x)./rho.^2 0; (R.*mu.*(2*m.*(2*eta_x.*(m.*rho_eta - m_eta.*rho) - eta_y.*(n.*rho_eta - n_eta.*rho) + 2*xi_x.*(m.*rho_xi - m_xi.*rho) - xi_y.*(n.*rho_xi - n_xi.*rho)) + 3*n.*(eta_x.*(n.*rho_eta - n_eta.*rho) + eta_y.*(m.*rho_eta - m_eta.*rho) + xi_x.*(n.*rho_xi - n_xi.*rho) + xi_y.*(m.*rho_xi - m_xi.*rho))) + R.*mu.*(2*m.*(4*eta_x.*(m.*rho_eta - m_eta.*rho) - 2*eta_y.*(n.*rho_eta - n_eta.*rho) + rho.*(2*eta_x.*m_eta - eta_y.*n_eta + 2*m_xi.*xi_x - n_

In [14]:
[(name, code)]  = codegen(('dGv_dU', dGv_dU), language='OCTAVE')
print(code)

function out1 = dGv_dU(R, e, e_eta, e_xi, eta_x, eta_y, gamma, k, m, m_eta, m_xi, mu, n, n_eta, n_xi, rho, rho_eta, rho_xi, xi_x, xi_y)
  %DGV_DU  Autogenerated by SymPy
  %   Code generated with SymPy 1.13.3
  %
  %   See http://www.sympy.org/ for more information.
  %
  %   This file is part of 'project'

  out1 = [0 0 0 0; mu.*(2*eta_x.*n.*rho_eta - eta_x.*n_eta.*rho + 2*eta_y.*m.*rho_eta - eta_y.*m_eta.*rho + 2*m.*rho_xi.*xi_y - m_xi.*rho.*xi_y + 2*n.*rho_xi.*xi_x - n_xi.*rho.*xi_x)./rho.^3 -mu.*(eta_y.*rho_eta + rho_xi.*xi_y)./rho.^2 -mu.*(eta_x.*rho_eta + rho_xi.*xi_x)./rho.^2 0; 2*mu.*(-2*eta_x.*m.*rho_eta + eta_x.*m_eta.*rho + 4*eta_y.*n.*rho_eta - 2*eta_y.*n_eta.*rho - 2*m.*rho_xi.*xi_x + m_xi.*rho.*xi_x + 4*n.*rho_xi.*xi_y - 2*n_xi.*rho.*xi_y)./(3*rho.^3) 2*mu.*(eta_x.*rho_eta + rho_xi.*xi_x)./(3*rho.^2) -4*mu.*(eta_y.*rho_eta + rho_xi.*xi_y)./(3*rho.^2) 0; (R.*mu.*(3*m.*(eta_x.*(n.*rho_eta - n_eta.*rho) + eta_y.*(m.*rho_eta - m_eta.*rho) + xi_x.*(n.*rho_xi - n_xi.*rho) + xi_

In [15]:
def generate(name, expr):
    substitutions, result_exprs = sympy.cse(expr)
    result_expr = result_exprs[-1]

    args = sorted(expr.free_symbols, key=str)

    lines = []
    lines.append(f"function {name} = {name}({', '.join(map(str, args))})")

    for var, var_expr in substitutions:
        lines.append("  " + octave_code(var_expr, assign_to=var))

    if hasattr(result_expr, "shape"):
        target = sympy.MatrixSymbol(name, *result_expr.shape)
    else:
        target = sympy.Symbol(name)

    lines.append("  " + octave_code(result_expr, assign_to=target))
    lines.append("end")

    return "\n".join(lines)

print(generate('dFv_dU', dFv_dU))
print(generate('dFv_dU_xi', dFv_dU_xi))
print(generate('dFv_dU_eta', dFv_dU_eta))
print(generate('dGv_dU', dGv_dU))
print(generate('dGv_dU_xi', dGv_dU_xi))
print(generate('dGv_dU_eta', dGv_dU_eta))
print(generate('A', A))
print(generate('B', B))

function dFv_dU = dFv_dU(R, e, e_eta, e_xi, eta_x, eta_y, gamma, k, m, m_eta, m_xi, mu, n, n_eta, n_xi, rho, rho_eta, rho_xi, xi_x, xi_y)
  x0 = n_eta.*rho;
  x1 = n_xi.*rho;
  x2 = eta_x.*rho_eta;
  x3 = 4*m;
  x4 = m_eta.*rho;
  x5 = eta_y.*rho_eta;
  x6 = 2*n;
  x7 = rho_xi.*xi_x;
  x8 = m_xi.*rho;
  x9 = rho_xi.*xi_y;
  x10 = rho.^(-3);
  x11 = mu.*x10;
  x12 = x2 + x7;
  x13 = rho.^2;
  x14 = 1./x13;
  x15 = mu.*x14;
  x16 = x12.*x15;
  x17 = x5 + x9;
  x18 = x15.*x17;
  x19 = 2*m;
  x20 = 1./R;
  x21 = m.*rho_eta - x4;
  x22 = eta_y.*x21;
  x23 = m.*rho_xi - x8;
  x24 = x23.*xi_y;
  x25 = n.*rho_eta - x0;
  x26 = eta_x.*x25;
  x27 = n.*rho_xi - x1;
  x28 = x27.*xi_x;
  x29 = x26 + x28;
  x30 = 3*n;
  x31 = eta_y.*x25;
  x32 = x27.*xi_y;
  x33 = eta_x.*x21;
  x34 = x23.*xi_x;
  x35 = 2*x33 + 2*x34;
  x36 = R.*mu;
  x37 = eta_x.*n_eta + n_xi.*xi_x;
  x38 = 2*x26 + 2*x28;
  x39 = eta_x.*m_eta;
  x40 = m_xi.*xi_x;
  x41 = -2*x31 - 2*x32 + 4*x33 + 4*x34;
  x42 = e_eta.*eta_x + e_xi.*x

In [16]:
from sympy.printing.rust import rust_code

def generate_rust(name, expr):
    substitutions, result_exprs = sympy.cse(
        expr,
        symbols=sympy.numbered_symbols("tmp")
    )

    result_expr = result_exprs[-1]
    args = sorted(expr.free_symbols, key=str)

    arg_list = ", ".join(f"{a}: f64" for a in args)

    is_matrix = isinstance(result_expr, sympy.MatrixBase)

    if is_matrix:
        rows, cols = result_expr.shape
        lines = [
            f"pub fn {name}({arg_list}) -> faer::Mat<f64> {{"
        ]
    else:
        lines = [
            f"pub fn {name}({arg_list}) -> f64 {{"
        ]

    for var, var_expr in substitutions:
        lines.append(f"    let {var} = {rust_code(var_expr)};")

    if is_matrix:
        lines.append(f"    let mut out = faer::Mat::<f64>::zeros({rows}, {cols});")

        for i in range(rows):
            for j in range(cols):
                rhs = rust_code(result_expr[i, j])
                lines.append(f"    out[({i}, {j})] = {rhs};")

        lines.append("    out")
    else:
        lines.append(f"    {rust_code(result_expr)}")

    lines.append("}")

    return "\n".join(lines)

print(generate_rust('dFv_dU', dFv_dU))
print(generate_rust('dFv_dU_xi', dFv_dU_xi))
print(generate_rust('dFv_dU_eta', dFv_dU_eta))
print(generate_rust('dGv_dU', dGv_dU))
print(generate_rust('dGv_dU_xi', dGv_dU_xi))
print(generate_rust('dGv_dU_eta', dGv_dU_eta))
print(generate_rust('A', A))
print(generate_rust('B', B))

pub fn dFv_dU(R: f64, e: f64, e_eta: f64, e_xi: f64, eta_x: f64, eta_y: f64, gamma: f64, k: f64, m: f64, m_eta: f64, m_xi: f64, mu: f64, n: f64, n_eta: f64, n_xi: f64, rho: f64, rho_eta: f64, rho_xi: f64, xi_x: f64, xi_y: f64) -> faer::Mat<f64> {
    let tmp0 = n_eta*rho;
    let tmp1 = n_xi*rho;
    let tmp2 = eta_x*rho_eta;
    let tmp3 = 4*m;
    let tmp4 = m_eta*rho;
    let tmp5 = eta_y*rho_eta;
    let tmp6 = 2*n;
    let tmp7 = rho_xi*xi_x;
    let tmp8 = m_xi*rho;
    let tmp9 = rho_xi*xi_y;
    let tmp10 = rho.powi(-3);
    let tmp11 = mu*tmp10;
    let tmp12 = tmp2 + tmp7;
    let tmp13 = rho.powi(2);
    let tmp14 = tmp13.recip();
    let tmp15 = mu*tmp14;
    let tmp16 = tmp12*tmp15;
    let tmp17 = tmp5 + tmp9;
    let tmp18 = tmp15*tmp17;
    let tmp19 = 2*m;
    let tmp20 = R.recip();
    let tmp21 = m*rho_eta - tmp4;
    let tmp22 = eta_y*tmp21;
    let tmp23 = m*rho_xi - tmp8;
    let tmp24 = tmp23*xi_y;
    let tmp25 = n*rho_eta - tmp0;
    let tmp26 = eta_x*tmp25;
  